In [1]:
from pathlib import Path
import os
import sys
import json
import csv

ROOT = Path.cwd()
if not (ROOT / "data" / "manifests").exists():
    ROOT = Path(r"D:\\FILE AND TASK\\TA")
assert (ROOT / "data" / "manifests").exists(), ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["MPLBACKEND"] = "Agg"

try:
    from IPython.display import display, Image
except ImportError:
    def display(item): print(item)
    class Image:
        def __init__(self, filename): self.filename = filename

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Project:", ROOT)
print("Kernel:", sys.executable)


def show_json(path):
    print(path)
    print(json.dumps(json.loads(path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))


def show_csv(path, rows=30):
    frame = pd.read_csv(path, low_memory=False)
    print(path, "|", len(frame), "baris")
    display(frame if len(frame) <= rows else pd.concat([frame.head(rows // 2), frame.tail(rows // 2)]))
    if len(frame) > rows:
        print("Tampilan adalah sampel; CSV lengkap tetap berada di path di atas.")
    return frame


Project: D:\FILE AND TASK\TA
Kernel: c:\Users\Fabio\AppData\Local\Programs\Python\Python314\python.exe


In [2]:
# Kode eksperimen berasal dari scripts/run_e1_clean_r0.py dan ditempatkan langsung di notebook ini.
# Tidak ada pemanggilan subprocess atau eksekusi file runner eksternal.
# Entry point CLI sengaja tidak dipanggil otomatis agar artefak valid tidak tertimpa.
__file__ = str(ROOT / "scripts" / "run_e1_clean_r0.py")

"""Run E1 clean retrieval for R0 only and write a self-contained artifact.

This command deliberately does not call the combined legacy evaluator, which
also starts E2 and E3.  It reads the active manifest and physical audio, then
writes one new timestamped artifact under ``results/reproductions``.
"""


import argparse
import hashlib
import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import librosa
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(__file__).resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.embeddings import AudioRepresentationExtractor
from src.preprocess import preprocess_audio
from src.retrieve import evaluate_retrieval_corpus

MANIFEST_PATH = PROJECT_ROOT / "data" / "manifests" / "dataset_split.csv"
OUTPUT_ROOT = PROJECT_ROOT / "results" / "reproductions"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def git_sha() -> str | None:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True, stderr=subprocess.DEVNULL
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return None


def extract_rows(extractor: AudioRepresentationExtractor, rows: pd.DataFrame, namespace: str) -> np.ndarray:
    embeddings: list[np.ndarray] = []
    total = len(rows)
    for position, (_, row) in enumerate(rows.iterrows(), start=1):
        audio = preprocess_audio(str(row["file_path"]))
        embedding = extractor.extract(audio, item_id=f"{namespace}:{row['recording_id']}")
        if embedding.shape != (40,) or not np.isfinite(embedding).all():
            raise ValueError(f"R0 embedding invalid for {row['recording_id']}: shape={embedding.shape}")
        embeddings.append(embedding.astype(np.float32, copy=False))
        if position % 25 == 0 or position == total:
            print(f"R0 {namespace}: {position}/{total}", flush=True)
    return np.stack(embeddings, axis=0)


def run(dry_run: bool = False) -> Path | None:
    manifest = pd.read_csv(MANIFEST_PATH)
    gallery = manifest.loc[manifest["split_role"].eq("gallery")].reset_index(drop=True)
    queries = manifest.loc[manifest["split_role"].eq("query_clean")].reset_index(drop=True)
    if len(gallery) != 673 or len(queries) != 311:
        raise ValueError(f"Active split mismatch: gallery={len(gallery)}, query_clean={len(queries)}")
    if gallery["recording_id"].duplicated().any() or queries["recording_id"].duplicated().any():
        raise ValueError("Duplicate recording_id found in active E1 split.")

    print(f"E1 R0 clean: gallery={len(gallery)}, query_clean={len(queries)}")
    if dry_run:
        return None

    extractor = AudioRepresentationExtractor("R0")
    gallery_embeddings = extract_rows(extractor, gallery, "gallery")
    query_embeddings = extract_rows(extractor, queries, "query_clean")

    summary, query_metrics = evaluate_retrieval_corpus(
        query_embs=query_embeddings,
        query_labels=queries["species_key"].tolist(),
        gallery_embs=gallery_embeddings,
        gallery_labels=gallery["species_key"].tolist(),
        k_values=[1, 5, 10],
    )
    per_query = pd.DataFrame(query_metrics)
    per_query.insert(0, "representation", "R0")
    per_query.insert(0, "author", queries["author"].astype(str))
    per_query.insert(0, "species_key", queries["species_key"].astype(str))
    per_query.insert(0, "query_id", queries["recording_id"].astype(str))

    now = datetime.now(timezone.utc)
    output_dir = OUTPUT_ROOT / f"E1_R0_{now.strftime('%Y%m%d_%H%M%S')}"
    output_dir.mkdir(parents=True, exist_ok=False)
    tables_dir = output_dir / "tables"
    features_dir = output_dir / "features"
    tables_dir.mkdir()
    features_dir.mkdir()
    np.savez_compressed(
        features_dir / "embeddings.npz",
        gallery_embeddings=gallery_embeddings,
        query_embeddings=query_embeddings,
        gallery_ids=gallery["recording_id"].astype(str).to_numpy(dtype=str),
        query_ids=queries["recording_id"].astype(str).to_numpy(dtype=str),
    )
    per_query.to_csv(tables_dir / "per_query.csv", index=False)
    metadata = {
        "artifact_schema": "e1_clean_r0_v2",
        "status": "complete",
        "representation": "R0",
        "experiment": "E1 Clean Retrieval only",
        "timestamp_utc": now.isoformat(),
        "git_sha": git_sha(),
        "manifest_sha256": sha256_file(MANIFEST_PATH),
        "preprocess_code_sha256": sha256_file(PROJECT_ROOT / "src" / "preprocess.py"),
        "mfcc_code_sha256": sha256_file(PROJECT_ROOT / "src" / "mfcc.py"),
        "retrieval_code_sha256": sha256_file(PROJECT_ROOT / "src" / "retrieve.py"),
        "python_version": sys.version.split()[0],
        "numpy_version": np.__version__,
        "librosa_version": librosa.__version__,
        "gallery_count": int(len(gallery)),
        "query_count": int(len(queries)),
        "summary": {key: float(value) for key, value in summary.items()},
    }
    (output_dir / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
    (output_dir / "README.md").write_text(
        "# E1 — Clean retrieval R0\n\n"
        "- `tables/per_query.csv`: metrik tiap kueri.\n"
        "- `features/embeddings.npz`: embedding gallery dan query.\n"
        "- `metadata.json`: konfigurasi dan ringkasan metrik.\n",
        encoding="utf-8",
    )
    print(json.dumps(summary, sort_keys=True))
    print(f"Artifact: {output_dir}")
    return output_dir

print("Kode run_e1_clean_r0.py telah dimuat di notebook. Jalankan fungsi eksperimen hanya jika memang ingin membuat run baru.")


Kode run_e1_clean_r0.py telah dimuat di notebook. Jalankan fungsi eksperimen hanya jika memang ingin membuat run baru.


In [3]:
# Kode eksperimen berasal dari scripts/materialize_e1_from_cache.py dan ditempatkan langsung di notebook ini.
# Tidak ada pemanggilan subprocess atau eksekusi file runner eksternal.
# Entry point CLI sengaja tidak dipanggil otomatis agar artefak valid tidak tertimpa.
__file__ = str(ROOT / "scripts" / "materialize_e1_from_cache.py")

"""Materialize auditable E1 artifacts from verified clean embedding caches.

This script never re-extracts audio embeddings.  It verifies the active split
against each cache, derives retrieval ranks and per-query metrics from the
cached vectors, and writes a new timestamped E1 artifact.  It is intended for
R1--R3 caches already used by the verified E4 run.
"""


import argparse
import hashlib
import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))

from src.embeddings import R3_CONTROL_VERSION
from src.retrieve import compute_query_retrieval, evaluate_retrieval_corpus

MANIFEST = ROOT / "data" / "manifests" / "dataset_split.csv"
OUTPUT_ROOT = ROOT / "results" / "reproductions"
CACHE_DIR = ROOT / "results" / "features"
DIMENSIONS = {"R0": 40, "R1": 2048, "R2": 1024, "R3": 40}


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def git_sha() -> str | None:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=ROOT, text=True, stderr=subprocess.DEVNULL
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return None


def load_and_validate_cache(rep: str, gallery: pd.DataFrame, queries: pd.DataFrame) -> tuple[Path, dict[str, np.ndarray]]:
    cache_path = CACHE_DIR / f"clean_embeddings_{rep}.npz"
    if not cache_path.exists():
        raise FileNotFoundError(f"Cache clean tidak ditemukan: {cache_path}")
    with np.load(cache_path) as raw:
        required = {"gal_feats", "qry_feats", "gal_ids", "qry_ids", "manifest_sha256"}
        missing = sorted(required - set(raw.files))
        if missing:
            raise ValueError(f"Cache {rep} tidak memiliki kolom wajib: {missing}")
        cache = {key: raw[key].copy() for key in raw.files}

    expected_manifest = sha256_file(MANIFEST)
    if str(cache["manifest_sha256"].item()) != expected_manifest:
        raise ValueError(f"Checksum manifest cache {rep} tidak cocok dengan manifest aktif.")
    if cache["gal_feats"].shape != (len(gallery), DIMENSIONS[rep]):
        raise ValueError(f"Bentuk gallery cache {rep} tidak cocok: {cache['gal_feats'].shape}")
    if cache["qry_feats"].shape != (len(queries), DIMENSIONS[rep]):
        raise ValueError(f"Bentuk query cache {rep} tidak cocok: {cache['qry_feats'].shape}")
    if not np.array_equal(cache["gal_ids"].astype(str), gallery["recording_id"].astype(str).to_numpy()):
        raise ValueError(f"Urutan gallery cache {rep} tidak cocok dengan manifest aktif.")
    if not np.array_equal(cache["qry_ids"].astype(str), queries["recording_id"].astype(str).to_numpy()):
        raise ValueError(f"Urutan query cache {rep} tidak cocok dengan manifest aktif.")
    if not np.isfinite(cache["gal_feats"]).all() or not np.isfinite(cache["qry_feats"]).all():
        raise ValueError(f"Cache {rep} mengandung embedding non-finite.")
    if rep == "R3":
        if "r3_control_version" not in cache or str(cache["r3_control_version"].item()) != R3_CONTROL_VERSION:
            raise ValueError("Versi kontrol R3 pada cache tidak cocok.")
    return cache_path, cache


def build_per_query(rep: str, gallery: pd.DataFrame, queries: pd.DataFrame, cache: dict[str, np.ndarray]) -> tuple[pd.DataFrame, dict]:
    gallery_vectors = cache["gal_feats"]
    query_vectors = cache["qry_feats"]
    gallery_labels = gallery["species_key"].astype(str).tolist()
    summary, metrics = evaluate_retrieval_corpus(
        query_vectors, queries["species_key"].astype(str).tolist(), gallery_vectors, gallery_labels
    )
    norm_gallery = gallery_vectors / np.maximum(np.linalg.norm(gallery_vectors, axis=1, keepdims=True), 1e-8)
    rows: list[dict] = []
    for index, (_, query) in enumerate(queries.iterrows()):
        vector = query_vectors[index]
        scores = norm_gallery @ (vector / max(float(np.linalg.norm(vector)), 1e-8))
        ranking = np.argsort(-scores, kind="stable")
        target = str(query["species_key"])
        target_mask = gallery["species_key"].astype(str).to_numpy() == target
        other_mask = ~target_mask
        top_index = int(ranking[0])
        true_max = float(np.max(scores[target_mask]))
        best_other = float(np.max(scores[other_mask]))
        row = {
            "query_id": str(query["recording_id"]),
            "species_key": target,
            "author": str(query["author"]),
            "representation": rep,
            "top1_recording_id": str(gallery.iloc[top_index]["recording_id"]),
            "top1_species_key": str(gallery.iloc[top_index]["species_key"]),
            "top1_similarity": float(scores[top_index]),
            "true_class_max_similarity": true_max,
            "best_other_similarity": best_other,
            "true_vs_other_margin": true_max - best_other,
        }
        row.update(metrics[index])
        rows.append(row)
    return pd.DataFrame(rows), {key: float(value) for key, value in summary.items()}


def materialize(rep: str) -> Path:
    manifest = pd.read_csv(MANIFEST, dtype={"recording_id": str})
    gallery = manifest.loc[manifest["split_role"].eq("gallery")].reset_index(drop=True)
    queries = manifest.loc[manifest["split_role"].eq("query_clean")].reset_index(drop=True)
    if len(gallery) != 673 or len(queries) != 311:
        raise ValueError(f"Split aktif tidak sesuai: gallery={len(gallery)}, query={len(queries)}")
    cache_path, cache = load_and_validate_cache(rep, gallery, queries)
    per_query, summary = build_per_query(rep, gallery, queries, cache)

    timestamp = datetime.now(timezone.utc)
    output = OUTPUT_ROOT / f"E1_{rep}_cache_{timestamp.strftime('%Y%m%dT%H%M%S_%fZ')}"
    tables = output / "tables"
    features = output / "features"
    output.mkdir(parents=True, exist_ok=False)
    tables.mkdir()
    features.mkdir()
    per_query.to_csv(tables / "per_query.csv", index=False)
    reference = {
        "source_cache_path": str(cache_path.relative_to(ROOT)).replace("\\", "/"),
        "source_cache_sha256": sha256_file(cache_path),
        "source_cache_manifest_sha256": str(cache["manifest_sha256"].item()),
        "embedding_dimension": DIMENSIONS[rep],
        "gallery_count": int(len(gallery)),
        "query_count": int(len(queries)),
    }
    (features / "cache_reference.json").write_text(json.dumps(reference, indent=2) + "\n", encoding="utf-8")
    metadata = {
        "artifact_schema": "e1_clean_cache_materialization_v1",
        "status": "complete",
        "experiment": "E1 Clean Retrieval",
        "representation": rep,
        "timestamp_utc": timestamp.isoformat(),
        "git_sha": git_sha(),
        "manifest_sha256": sha256_file(MANIFEST),
        "retrieval_code_sha256": sha256_file(ROOT / "src" / "retrieve.py"),
        "cache_provenance": reference,
        "summary": summary,
    }
    if rep == "R3":
        metadata["r3_control_version"] = R3_CONTROL_VERSION
    (output / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
    (output / "README.md").write_text(
        f"# E1 clean retrieval {rep}\n\n"
        "Artefak ini dihitung ulang dari cache embedding bersih yang diverifikasi; "
        "audio dan embedding tidak diekstrak ulang. `tables/per_query.csv` berisi "
        "peringkat, prediksi Top-1, serta metrik retrieval tiap query.\n",
        encoding="utf-8",
    )
    print(f"Artifact: {output}")
    print(json.dumps(summary, sort_keys=True))
    return output

print("Kode materialize_e1_from_cache.py telah dimuat di notebook. Jalankan fungsi eksperimen hanya jika memang ingin membuat run baru.")


Kode materialize_e1_from_cache.py telah dimuat di notebook. Jalankan fungsi eksperimen hanya jika memang ingin membuat run baru.


In [4]:
runs = {
    "R0": ROOT / "results/reproductions/E1_R0_20261011_175321",
    "R1": ROOT / "results/reproductions/E1_R1_cache_20261012T021032_566409Z",
    "R2": ROOT / "results/reproductions/E1_R2_cache_20261012T021034_128124Z",
    "R3": ROOT / "results/reproductions/E1_R3_cache_20261012T021035_119528Z",
}
summary_rows = []
for representation, path in runs.items():
    metadata = json.loads((path / "metadata.json").read_text(encoding="utf-8"))
    summary_rows.append({"representation": representation, **metadata["summary"]})
summary = pd.DataFrame(summary_rows).sort_values("representation")
display(summary)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
summary.plot.bar(x="representation", y="mAP@10", ax=axes[0], legend=False, color="#356f9b", title="E1 Clean mAP@10")
summary.plot.bar(x="representation", y="mean_top1", ax=axes[1], legend=False, color="#5a9d6f", title="E1 Clean Top-1 accuracy")
for axis in axes: axis.set_ylim(0, 1)
fig.tight_layout(); display(fig); plt.close(fig)
for representation, path in runs.items():
    show_csv(path / "tables" / "per_query.csv", rows=20)
print("R0 adalah MFCC baseline. Preprocessing adalah tahap terpisah pada notebook 01.")


,representation,num_queries,mean_top1,MRR,mAP@1,Recall@1,Precision@1,mAP@5,Recall@5,Precision@5,mAP@10,Recall@10,Precision@10
0,R0,311.0,0.183280,0.310923,0.183280,0.006187,0.183280,0.102680,0.022114,0.153055,0.069393,0.036321,0.131190
1,R1,311.0,0.376206,0.507048,0.376206,0.013406,0.376206,0.231726,0.048854,0.294534,0.176268,0.084175,0.258521
2,R2,311.0,0.816720,0.865158,0.816720,0.028796,0.816720,0.711919,0.128696,0.746624,0.636709,0.233109,0.693891
3,R3,311.0,0.064309,0.163871,0.064309,0.001586,0.064309,0.030975,0.007332,0.058521,0.021141,0.014092,0.056913


<Figure size 1200x450 with 2 Axes>

D:\FILE AND TASK\TA\results\reproductions\E1_R0_20261011_175321\tables\per_query.csv | 311 baris


,query_id,species_key,author,representation,max_score,top1_match,RR,total_relevant_in_gallery,P@1,R@1,AP@1,P@5,R@5,AP@5,P@10,R@10,AP@10
0,XC42472,savnig1,David Marques,R0,0.991959,1,1.000000,15,1.0,0.066667,1.0,0.4,0.133333,0.280000,0.2,0.133333,0.140000
1,XC1098429,ashtai1,Yasinchumaedi,R0,0.991454,0,0.010753,18,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000
2,XC20097,ashtai1,Sander Pieterse & Ben Wielstra,R0,0.994453,0,0.013158,18,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000
3,XC20098,ashtai1,Sander Pieterse & Ben Wielstra,R0,0.992043,0,0.010000,18,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000
4,XC214420,ashtai1,Paulo Alves,R0,0.996303,0,0.011628,18,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000
5,XC236330,ashtai1,Paulo Alves,R0,0.985306,0,0.012658,18,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000
6,XC34503,ashtai1,Tjalle Boorsma,R0,0.979483,0,0.200000,18,0.0,0.000000,0.0,0.2,0.055556,0.040000,0.1,0.055556,0.020000
7,XC151815,colkin1,Mike Nelson,R0,0.994588,0,0.090909,19,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000
8,XC152005,colkin1,Mike Nelson,R0,0.984806,0,0.038462,19,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000
9,XC152203,colkin1,Mike Nelson,R0,0.992526,1,1.000000,19,1.0,0.052632,1.0,0.2,0.052632,0.200000,0.1,0.052632,0.100000


Tampilan adalah sampel; CSV lengkap tetap berada di path di atas.
D:\FILE AND TASK\TA\results\reproductions\E1_R1_cache_20261012T021032_566409Z\tables\per_query.csv | 311 baris


...

Tampilan adalah sampel; CSV lengkap tetap berada di path di atas.
D:\FILE AND TASK\TA\results\reproductions\E1_R2_cache_20261012T021034_128124Z\tables\per_query.csv | 311 baris


,query_id,species_key,author,representation,top1_recording_id,top1_species_key,top1_similarity,true_class_max_similarity,best_other_similarity,true_vs_other_margin,...,total_relevant_in_gallery,P@1,R@1,AP@1,P@5,R@5,AP@5,P@10,R@10,AP@10
0,XC42472,savnig1,David Marques,R2,XC954966,savnig1,0.800407,0.800407,0.669977,0.130429,...,15,1.0,0.066667,1.0,1.0,0.333333,1.000000,1.0,0.666667,1.000000
1,XC1098429,ashtai1,Yasinchumaedi,R2,XC208015,ashtai1,0.631527,0.631527,0.587906,0.043621,...,18,1.0,0.055556,1.0,0.8,0.222222,0.800000,0.7,0.388889,0.646825
2,XC20097,ashtai1,Sander Pieterse & Ben Wielstra,R2,XC208015,ashtai1,0.646827,0.646827,0.614792,0.032035,...,18,1.0,0.055556,1.0,0.4,0.111111,0.400000,0.2,0.111111,0.200000
3,XC20098,ashtai1,Sander Pieterse & Ben Wielstra,R2,XC208014,ashtai1,0.669163,0.669163,0.644337,0.024826,...,18,1.0,0.055556,1.0,0.8,0.222222,0.643333,0.8,0.444444,0.667103
4,XC214420,ashtai1,Paulo Alves,R2,XC168880,hoopit1,0.749145,0.714433,0.749145,-0.034712,...,18,0.0,0.000000,0.0,0.8,0.222222,0.543333,0.4,0.222222,0.271667
5,XC236330,ashtai1,Paulo Alves,R2,XC208015,ashtai1,0.678046,0.678046,0.644060,0.033986,...,18,1.0,0.055556,1.0,0.8,0.222222,0.800000,0.7,0.388889,0.646825
6,XC34503,ashtai1,Tjalle Boorsma,R2,XC208014,ashtai1,0.708441,0.708441,0.654326,0.054115,...,18,1.0,0.055556,1.0,0.8,0.222222,0.800000,0.8,0.444444,0.745437
7,XC151815,colkin1,Mike Nelson,R2,XC286713,colkin1,0.675565,0.675565,0.652455,0.023110,...,19,1.0,0.052632,1.0,0.6,0.157895,0.600000,0.7,0.368421,0.574762
8,XC152005,colkin1,Mike Nelson,R2,XC69653,colkin1,0.717367,0.717367,0.605016,0.112351,...,19,1.0,0.052632,1.0,1.0,0.263158,1.000000,0.8,0.421053,0.776389
9,XC152203,colkin1,Mike Nelson,R2,XC69228,pygcup1,0.578650,0.528579,0.578650,-0.050071,...,19,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000


Tampilan adalah sampel; CSV lengkap tetap berada di path di atas.
D:\FILE AND TASK\TA\results\reproductions\E1_R3_cache_20261012T021035_119528Z\tables\per_query.csv | 311 baris


,query_id,species_key,author,representation,top1_recording_id,top1_species_key,top1_similarity,true_class_max_similarity,best_other_similarity,true_vs_other_margin,...,total_relevant_in_gallery,P@1,R@1,AP@1,P@5,R@5,AP@5,P@10,R@10,AP@10
0,XC42472,savnig1,David Marques,R3,XC287770,subwar1,0.410280,0.168684,0.410280,-0.241597,...,15,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000
1,XC1098429,ashtai1,Yasinchumaedi,R3,XC67798,rubcuc1,0.497373,0.323404,0.497373,-0.173969,...,18,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000
2,XC20097,ashtai1,Sander Pieterse & Ben Wielstra,R3,XC255441,rubcuc1,0.496470,0.432224,0.496470,-0.064246,...,18,0.0,0.0,0.0,0.2,0.055556,0.066667,0.2,0.111111,0.055556
3,XC20098,ashtai1,Sander Pieterse & Ben Wielstra,R3,XC596830,rubcuc1,0.456066,0.235471,0.456066,-0.220595,...,18,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000
4,XC214420,ashtai1,Paulo Alves,R3,XC746052,olbsun1,0.433755,0.317963,0.433755,-0.115791,...,18,0.0,0.0,0.0,0.0,0.000000,0.000000,0.1,0.055556,0.010000
5,XC236330,ashtai1,Paulo Alves,R3,XC1059449,colkin1,0.482709,0.225218,0.482709,-0.257491,...,18,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000
6,XC34503,ashtai1,Tjalle Boorsma,R3,XC89580,moutai1,0.447562,0.320675,0.447562,-0.126887,...,18,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000
7,XC151815,colkin1,Mike Nelson,R3,XC1059451,comior1,0.439714,0.362040,0.439714,-0.077675,...,19,0.0,0.0,0.0,0.2,0.052632,0.050000,0.1,0.052632,0.025000
8,XC152005,colkin1,Mike Nelson,R3,XC206313,pygcup1,0.466805,0.259593,0.466805,-0.207212,...,19,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000
9,XC152203,colkin1,Mike Nelson,R3,XC940265,pygcup1,0.496121,0.184617,0.496121,-0.311504,...,19,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000


Tampilan adalah sampel; CSV lengkap tetap berada di path di atas.
R0 adalah MFCC baseline. Preprocessing adalah tahap terpisah pada notebook 01.
